In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ANES Trust Data

## Raw Data Import

In [2]:
# 1948-2020 raw data (no district level information for 2020)
file_path_1 = "data/raw/anes/anes_timeseries_cdf_csv_20220916.csv"
df1 = pd.read_csv(file_path_1, low_memory=False)

In [3]:
# Drop irrelevant columns
columns_to_keep = [
    "VCF0004",    # Year
    
    "VCF0900b",   # State-District Code
    "VCF0900",    # District Code
    "VCF0901b",    # State Fips Code
    
    # Dependent (trust) variables
    "VCF0604",    
    "VCF0605",
    "VCF0606",
    "VCF0608",
    "VCF0656",
    
    # Independent variables
    "VCF0101",    # Age
    "VCF0070a",   # Gender
    "VCF0071c",   # Race
    "VCF0072a",   # Ethnicity
    "VCF0140a",   # Education
    "VCF0114",    # Income percentile
    "VCF0116",    # Employment status
    "VCF0302",    # Political affiliation
    "VCF0846",    # Religion importance
]

df1 = df1[columns_to_keep]

In [4]:
# Rename columns for clarity
df1 = df1.rename(columns={
    "VCF0004": "year",
    "VCF0900b": "state_district_code",
    "VCF0901b": "state_code",
    "VCF0900": "district_code",
    
    "VCF0604": "trust_gov_right",
    "VCF0605": "gov_for_all",
    "VCF0606": "gov_waste",
    "VCF0608": "crooked_officials",
    "VCF0656": "trust_gov_index",
    
    "VCF0101": "age",
    "VCF0070a": "gender",
    "VCF0071c": "race",
    "VCF0072a": "ethnicity",
    "VCF0140a": "education",
    "VCF0114": "income",
    "VCF0116": "employment_status",
    "VCF0302": "political_affiliation",
    "VCF0846": "religion_importance",
})

# Clean trust_gov_right for adjustment
df1['trust_gov_right'] = pd.to_numeric(df1['trust_gov_right'], errors='coerce')
df1['trust_gov_right'] = df1['trust_gov_right'].replace([-9, -8, -1, 0, 9], np.nan)
df1["trust_gov_right"] = df1["trust_gov_right"].replace(
    { 
        4: 5,  
        3: 4,   
    }
) # Adjust order to match scale of 2024

In [5]:
# 2024 raw data (no district-level information)
file_path_2 = "data/raw/anes/anes_timeseries_2024_csv_20250430.csv"
df2 = pd.read_csv(file_path_2, low_memory=False)

In [6]:
# Drop irrelevant columns
columns_to_keep_2 = [
    "V241229",
    "V241231",
    "V241232",
]

df2 = df2[columns_to_keep_2]

In [7]:
# Rename columns for clarity
df2 = df2.rename(columns={
    "V241229": "trust_gov_right",
    "V241231": "gov_for_all",
    "V241232": "gov_waste",
})

# Adjust order to match with older data
df2["trust_gov_right"] = df2["trust_gov_right"].replace(
    {
        5: 1,  
        4: 2,  
        3: 3,  
        2: 4,  
        1: 5  
    }
)

df2["year"] = 2024

## Merge ANES Datasets

In [8]:
# Merge datasets
df_trust = pd.concat([df1, df2], ignore_index=True).sort_values(by="year").reset_index(drop=True)
for col in df_trust.columns:
    if col != 'state_code':
        df_trust[col] = pd.to_numeric(df_trust[col], errors='coerce')

In [9]:
# Drop invalid location variables
df_trust['state_district_code'] = df_trust['state_district_code'].replace([9999,1100], pd.NA) # 1100 is one DC observation with error
df_trust['district_code'] = df_trust['district_code'].replace([0, 99], [pd.NA, 98])
df_trust['state_code'] = df_trust['state_code'].replace(['99'], pd.NA)
df_trust["state_code"] = (
    df_trust["state_code"]
    .astype(str)          # Convert to string first
    .replace(["<NA>", "nan", "NA", ""], np.nan)  # Unify missing values
)

In [10]:
# Set state_district_code to four digit string
df_trust['state_district_code'] = (
    df_trust['state_district_code']
    .dropna()
    .astype(int)
    .astype(str)
    .str.zfill(4)
    .reindex(df_trust.index)
)

# Set district code to two digit string
df_trust['district_code'] = (
    df_trust['district_code']
    .dropna()
    .astype(int)
    .astype(str)
    .str.zfill(2)
    .reindex(df_trust.index)
)

In [11]:
# Adjust district code for at-large states
at_large = {"AK", "DE", "ND", "VT", "SD", "WY"}
mask = df_trust["state_code"].isin(at_large)

df_trust.loc[mask, "district_code"] = "00"
df_trust.loc[mask, "state_district_code"] = (
    df_trust.loc[mask, "state_district_code"].str[:2] + "00"
)


In [12]:
# Drop invalid trust variables
trust_var_no_index = [    
    "trust_gov_right",
    "gov_for_all",
    "gov_waste",
    "crooked_officials",
]

df_trust[trust_var_no_index] = df_trust[trust_var_no_index].replace([-9,-8,-1,0,9], pd.NA)
df_trust['trust_gov_index'] = df_trust['trust_gov_index'].replace([999], pd.NA)

In [13]:
# Drop invalid independent variables
independent_var = [  
    "age",
    "gender",
    "race",
    "ethnicity",
    "education",
    "income",
    "employment_status",
    "political_affiliation",
    "religion_importance",
]
df_trust[independent_var] = df_trust[independent_var].replace([0,9], pd.NA)
df_trust['race'] = df_trust['race'].replace([7], pd.NA)
df_trust['ethnicity'] = df_trust['ethnicity'].replace([7], pd.NA)
df_trust['education'] = df_trust['education'].replace([8], pd.NA)
df_trust['political_affiliation'] = df_trust['political_affiliation'].replace([8], pd.NA)
df_trust['religion_importance'] = df_trust['religion_importance'].replace([8], pd.NA)

In [14]:
# Make missing variables consistent
loc_cols = {
    "state_district_code",
    "state_code",
    "district_code",
}

for col in df_trust.columns:
    if col not in loc_cols:
        df_trust[col] = pd.to_numeric(df_trust[col], errors="coerce")


## Trust Variable Transformation

In [15]:
# Define valid ranges for each variable
valid_ranges = {
    # Trust variables
    'trust_gov_right': (1, 5), 
    'gov_for_all': (1, 2),
    'gov_waste': (1, 3),
    'crooked_officials': (1, 3),
    'trust_gov_index': (0, 100),
}

# Z-score standardization for each trust variable
scaler = StandardScaler()

for col, (min_val, max_val) in valid_ranges.items():
        # Create mask for valid values
        valid_mask = df_trust[col].between(min_val, max_val, inclusive='both')
        
        # Standardize only valid values
        if valid_mask.any():
            df_trust.loc[valid_mask, f'{col}_z'] = scaler.fit_transform(
                df_trust.loc[valid_mask, [col]]
            )
        else:
            df_trust[f'{col}_z'] = np.nan  # Handle all-invalid columns

In [16]:
# Dummy variables
df_trust['trust_gov_right_d'] = df_trust['trust_gov_right'].apply(
    lambda x: 1 if x in [4, 5] else (x if pd.isna(x) else 0)
)

df_trust['gov_for_all_d'] = df_trust['gov_for_all'].apply(
    lambda x: 1 if x == 2 else (x if pd.isna(x) else 0)
)

df_trust['gov_waste_d'] = df_trust['gov_waste'].apply(
    lambda x: 1 if x in [2, 3] else (x if pd.isna(x) else 0)
)

df_trust['crooked_officials_d'] = df_trust['crooked_officials'].apply(
    lambda x: 1 if x in [2, 3] else (x if pd.isna(x) else 0)
)

df_trust['trust_gov_index_d'] = df_trust['trust_gov_index'].apply(
    lambda x: 1 if x > 50 else (x if pd.isna(x) else 0)
)

In [17]:
# Save an individual-level ANES dataset for times series
df_trust.to_csv("data/intermediate/anes_individual_level.csv")

## Collapse to District Level and Export

In [18]:
# Variable categories
trust_vars = [    
    "trust_gov_right",
    "gov_for_all",
    "gov_waste",
    "crooked_officials",
    "trust_gov_index",

    "trust_gov_right_z",
    "gov_for_all_z",
    "gov_waste_z",
    "crooked_officials_z",
    "trust_gov_index_z",

    "trust_gov_right_d",
    "gov_for_all_d",
    "gov_waste_d",
    "crooked_officials_d",
    "trust_gov_index_d",
]

independent_vars = [  
    "age",
    "gender",
    "race",
    "ethnicity",
    "education",
    "income",
    "employment_status",
    "political_affiliation",
    "religion_importance",
]

loc_vars = [
    "district_code",
    "state_code",
]

# Combine all variables by aggregation type
agg_dict = {
    **{var: 'first' for var in loc_vars},
    **{var: 'mean' for var in trust_vars + independent_vars}
}

# Group and aggregate
df_trust = df_trust.groupby(['year', 'state_district_code'], as_index=False).agg(agg_dict)

In [19]:
# Rename variables
df_trust = df_trust.rename(columns={
    "age": "mean_age",
    "gender": "perc_female",
    "race": "perc_nonwhite",
    "ethnicity": "perc_hispanic",
    "education": "mean_education_level",
    "income": "mean_income_level",
    "employment_status": "employed_rate",
    "political_affiliation": "left_leaning_level",
    "religion_importance": "mean_religion_importance_level",
})

In [20]:
# Save an district-level ANES dataset for heat map
df_trust.to_csv("data/intermediate/anes_district_level.csv")